In [0]:
from pyspark.sql import functions as F

df = spark.read.parquet(
    '/Volumes/blueprint_spark/bronze/bronze/vendas.parquet',
    mergeSchema=True,
)

# Pasta onde as aulas vão gravar
base = "/Volumes/blueprint_spark/bronze/bronze/modulo07"

In [0]:
df.write.parquet(f'{base}/vendas_parquet')
df.write.csv(f'{base}/vendas_csv',header=True)
df.write.json(f'{base}/vendas_json')

In [0]:
spark.read.parquet(f"{base}/vendas_parquet").show(5)


+---+-----------+--------+--------+--------+------+---+----------+
| id|vendedor_id|vendedor|  regiao| produto| valor|mes|data_venda|
+---+-----------+--------+--------+--------+------+---+----------+
|  1|          1|     Ana| Sudeste|Notebook|3200.0|Jan|2026-01-05|
|  2|          1|     Ana| Sudeste|   Mouse|  80.0|Jan|2026-01-07|
|  3|          2|   Bruno|     Sul| Monitor| 950.0|Jan|2026-01-10|
|  4|          2|   Bruno|     Sul| Teclado| 220.0|Fev|2026-02-02|
|  5|          3|   Carla|Nordeste|Notebook|3400.0|Fev|2026-02-11|
+---+-----------+--------+--------+--------+------+---+----------+
only showing top 5 rows


In [0]:
spark.read.csv(f"{base}/vendas_csv", header=True).show(5)

+---+-----------+--------+--------+--------+------+---+----------+
| id|vendedor_id|vendedor|  regiao| produto| valor|mes|data_venda|
+---+-----------+--------+--------+--------+------+---+----------+
|  1|          1|     Ana| Sudeste|Notebook|3200.0|Jan|2026-01-05|
|  2|          1|     Ana| Sudeste|   Mouse|  80.0|Jan|2026-01-07|
|  3|          2|   Bruno|     Sul| Monitor| 950.0|Jan|2026-01-10|
|  4|          2|   Bruno|     Sul| Teclado| 220.0|Fev|2026-02-02|
|  5|          3|   Carla|Nordeste|Notebook|3400.0|Fev|2026-02-11|
+---+-----------+--------+--------+--------+------+---+----------+
only showing top 5 rows


In [0]:
df.count()

15

In [0]:
df.write.mode('overwrite').parquet(f'{base}/vendas_parquet')

In [0]:
spark.read.parquet(f"{base}/vendas_parquet").count()


15

In [0]:
display(df)

id,vendedor_id,vendedor,regiao,produto,valor,mes,data_venda
1,1,Ana,Sudeste,Notebook,3200.0,Jan,2026-01-05
2,1,Ana,Sudeste,Mouse,80.0,Jan,2026-01-07
3,2,Bruno,Sul,Monitor,950.0,Jan,2026-01-10
4,2,Bruno,Sul,Teclado,220.0,Fev,2026-02-02
5,3,Carla,Nordeste,Notebook,3400.0,Fev,2026-02-11
6,3,Carla,Nordeste,Webcam,150.0,Fev,2026-02-14
7,4,Diego,Centro-Oeste,Monitor,980.0,Mar,2026-03-03
8,4,Diego,Centro-Oeste,Notebook,3100.0,Mar,2026-03-09
9,5,Elaine,Norte,Mouse,75.0,Mar,2026-03-15
10,5,Elaine,Norte,Teclado,210.0,Mar,2026-03-20


In [0]:
df.write.mode("overwrite").partitionBy("regiao").parquet(f"{base}/particao_regiao")

In [0]:
spark.read.parquet(f"{base}/particao_regiao").filter(F.col("regiao")=="Sul").show()

+---+-----------+--------+--------+------+---+----------+------+
| id|vendedor_id|vendedor| produto| valor|mes|data_venda|regiao|
+---+-----------+--------+--------+------+---+----------+------+
| 12|          2|   Bruno|Notebook|3300.0|Abr|2026-04-08|   Sul|
|  4|          2|   Bruno| Teclado| 220.0|Fev|2026-02-02|   Sul|
|  3|          2|   Bruno| Monitor| 950.0|Jan|2026-01-10|   Sul|
+---+-----------+--------+--------+------+---+----------+------+



In [0]:
df.write.mode("overwrite").partitionBy("regiao","mes").parquet(f"{base}/particao_regiao_mes")

In [0]:
# Overwrite com partição: por padrão o overwrite apaga TODAS as partições.
# Com partitionOverwriteMode = dynamic, só as partições presentes nos dados novos são substituídas
df.filter(F.col("regiao") == "Sul") \
  .write.mode("overwrite") \
  .partitionBy("regiao") \
  .parquet(f"{base}/particao_regiao")
display(dbutils.fs.ls(f"{base}/particao_regiao"))   # as outras regiões continuam lá

path,name,size,modificationTime
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/particao_regiao/_SUCCESS,_SUCCESS,0,1793488427000
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/particao_regiao/_committed_6885843723265115315,_committed_6885843723265115315,35,1793488426000
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/particao_regiao/regiao=Centro-Oeste/,regiao=Centro-Oeste/,0,1793488426688
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/particao_regiao/regiao=Nordeste/,regiao=Nordeste/,0,1793488426688
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/particao_regiao/regiao=Norte/,regiao=Norte/,0,1793488426688
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/particao_regiao/regiao=Sudeste/,regiao=Sudeste/,0,1793488426688
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/particao_regiao/regiao=Sul/,regiao=Sul/,0,1793488426688


In [0]:
caminho = "/Volumes/blueprint_spark/bronze/bronze/modulo07/organizacao"

def qtd_arquivos(caminho):
    return len([f for f in dbutils.fs.ls(caminho) if f.name.endswith(".parquet")])

In [0]:
# 1. Escrita normal: o Spark decide quantos arquivos gerar
df.write.mode("overwrite").parquet(f"{base}/org_padrao")
print("Padrão:", qtd_arquivos(f"{base}/org_padrao"))

Padrão: 1


In [0]:
# 2. coalesce(n): JUNTA partições, sem shuffle. Usado para REDUZIR a quantidade de arquivos
df.coalesce(2).write.mode("overwrite").parquet(f"{base}/org_coalesce")
print("coalesce(1):", qtd_arquivos(f"{base}/org_coalesce"))

coalesce(1): 1


In [0]:
# 3. repartition(n): REDISTRIBUI os dados com shuffle. Aumenta ou diminui, com arquivos de tamanho parecido
df.repartition(4).write.mode("overwrite").parquet(f"{base}/org_repartition")
print("repartition(4):", qtd_arquivos(f"{base}/org_repartition"))


repartition(4): 4


In [0]:

# 4. repartition pela mesma coluna do partitionBy: 1 arquivo por pasta, em vez de vários
df.repartition("regiao") \
  .write.mode("overwrite").partitionBy("regiao").parquet(f"{base}/org_regiao")
print("Arquivos em regiao=Sul:", qtd_arquivos(f"{base}/org_regiao/regiao=Sul"))

Arquivos em regiao=Sul: 1


In [0]:
# 5. Compressão: snappy é o padrão do parquet (rápido). gzip comprime mais, mas é mais lento
df.write.mode("overwrite").parquet(f"{base}/org_gzip", compression="gzip")
display(dbutils.fs.ls(f"{base}/org_gzip"))

path,name,size,modificationTime
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/org_gzip/_SUCCESS,_SUCCESS,0,1793489473000
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/org_gzip/_committed_7748829864755056113,_committed_7748829864755056113,120,1793489473000
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/org_gzip/_started_7748829864755056113,_started_7748829864755056113,0,1793489472000
dbfs:/Volumes/blueprint_spark/bronze/bronze/modulo07/org_gzip/part-00000-tid-7748829864755056113-0216ee11-5868-4e50-935b-3ab6299688e8-341-1-c000.gz.parquet,part-00000-tid-7748829864755056113-0216ee11-5868-4e50-935b-3ab6299688e8-341-1-c000.gz.parquet,3321,1793489473000


In [0]:
from pyspark.sql import functions as F

# DataFrame de 1 milhão de linhas gerado em código
cidades = F.array(*[F.lit(c) for c in ["Salvador", "São Paulo", "Rio de Janeiro", "Recife", "Fortaleza", "Curitiba"]])
produtos = F.array(*[F.lit(p) for p in ["Notebook", "Mouse", "Monitor", "Teclado", "Webcam"]])

df_grande = (
    spark.range(1_000_000)                                                         # coluna id de 0 a 999.999
    .withColumn("cidade", F.element_at(cidades, (F.col("id") % 6 + 1).cast("int")))
    .withColumn("produto", F.element_at(produtos, (F.col("id") % 5 + 1).cast("int")))
    .withColumn("valor", F.round(F.rand(seed=42) * 3000 + 50, 2))
    .withColumn("data_venda", F.date_add(F.lit("2026-01-01"), (F.rand(seed=7) * 270).cast("int")))
)

df_grande.show(5)
print("Linhas:", df_grande.count())

# Tamanho total dos arquivos .parquet de uma pasta, em MB
def tamanho_mb(caminho):
    return round(sum(f.size for f in dbutils.fs.ls(caminho) if f.name.endswith(".parquet")) / 1024 / 1024, 2)

# Mesmo DataFrame, 1 arquivo (coalesce(1)), mudando só a compressão
for compressao in ["none", "snappy", "gzip"]:
    caminho = f"{base}/compressao_{compressao}"
    df_grande.coalesce(1).write.mode("overwrite").parquet(caminho, compression=compressao)
    print(f"{compressao:>7}: {tamanho_mb(caminho)} MB")

# Esperado: none > snappy > gzip
# snappy (padrão) = bom equilíbrio entre tamanho e velocidade
# gzip            = arquivo menor, porém escrita e leitura mais lentas


+---+--------------+--------+------+----------+
| id|        cidade| produto| valor|data_venda|
+---+--------------+--------+------+----------+
|  0|      Salvador|Notebook|307.27|2026-09-20|
|  1|     São Paulo|   Mouse|981.23|2026-03-06|
|  2|Rio de Janeiro| Monitor|237.71|2026-07-06|
|  3|        Recife| Teclado|969.38|2026-07-13|
|  4|     Fortaleza|  Webcam| 50.13|2026-07-31|
+---+--------------+--------+------+----------+
only showing top 5 rows
Linhas: 1000000
   none: 17.06 MB
 snappy: 9.79 MB
   gzip: 5.78 MB
